# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 4 — Embeddings e ingesta documental

ITBA Educación Ejecutiva

Preparación de información para recuperación semántica: hoy dejamos de hablarle al modelo de a una consulta por vez, y empezamos a construir el corpus sobre el que va a buscar.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo (demo con AWS Bedrock). No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 9 (Laboratorio)**: tarea principal de la semana — checkpoint del proyecto integrador. Se resuelve con el proveedor de cada uno.

## Objetivo de la clase

Al final de la clase vas a poder:
- Explicar qué es un embedding y por qué la similitud coseno mide "cercanía de significado".
- Limpiar y normalizar texto antes de vectorizarlo.
- Elegir una estrategia de chunking (tamaño, solapamiento) con criterio, no por default.
- Diseñar metadata que permita trazar cada fragmento hasta su documento de origen.
- Separar claramente el pipeline de ingesta del de consulta.

**Checkpoint del proyecto integrador de hoy**: pipeline de ingesta del corpus (limpieza, chunking y metadata) — ver `proyecto-integrador/README.md`, checkpoint Clase 4.

## Agenda

1. Información estructurada vs. no estructurada
2. Embeddings y similitud semántica
3. Tu proveedor: embeddings — referencia rápida
4. Limpieza y normalización
5. Estrategias de chunking y solapamiento
6. Metadata, identificadores y trazabilidad
7. Separación entre ingesta y consulta
8. Comparación de estrategias de chunking
9. Chunking semántico (contenido complementario)
10. Laboratorio: pipeline de ingesta documental

## 1. Información estructurada vs. no estructurada

- **Estructurada**: tablas, JSON, filas de una base de datos — ya tiene un esquema. Se consulta con SQL/filtros exactos, no necesita embeddings.
- **No estructurada**: texto libre (documentación, tickets, emails, PDFs) — no tiene un esquema uniforme. Es el caso que nos ocupa de acá en adelante: para "buscar por significado" en texto libre necesitamos convertirlo a una representación que capture semántica, no solo keywords.

El corpus del TP integrador (documentación técnica) es no estructurado — eso es justamente lo que hace necesario todo lo que viene en esta clase.

## 2. Embeddings y similitud semántica

Un embedding es un vector $\vec{v} \in \mathbb{R}^d$ que representa el significado de un texto. Textos semánticamente parecidos quedan "cerca" en ese espacio — se mide con **similitud coseno**:

$$
\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \, \lVert \vec{b} \rVert}
$$

Cercano a 1 → misma dirección → mismo significado aproximado. Cercano a 0 → sin relación. (Esto ya se mencionó en la Clase 1 — hoy lo usamos en código real por primera vez.)

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
    """Devuelve un embedding (lista de floats) por cada texto de la lista, usando Cohere Embed v4 en Bedrock."""
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    resp_body = json.loads(resp["body"].read())
    return resp_body["embeddings"]["float"]

In [ ]:
import numpy as np

def similitud_coseno(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

frases = [
    "La ventana de contexto limita cuánto texto puede procesar un modelo por llamada.",
    "El límite de tokens de entrada y salida define cuánta información cabe en una consulta.",
    "Ualá es una fintech argentina que ofrece una tarjeta prepaga y una cuenta digital.",
]

vectores = embed_bedrock(frases)
print(f"Dimensión de cada embedding: {len(vectores[0])}")

print("\nSimilitud entre frase 1 y 2 (ambas sobre contexto/tokens):", round(similitud_coseno(vectores[0], vectores[1]), 3))
print("Similitud entre frase 1 y 3 (temas no relacionados):       ", round(similitud_coseno(vectores[0], vectores[2]), 3))

**🏠 Practicá**: agregá una cuarta frase que hable de otra cosa totalmente distinta (ej. una receta de cocina) y confirmá que su similitud con las otras tres es la más baja del grupo.

## 3. Tu proveedor: embeddings — referencia rápida

Igual que con generación (Clase 2), elegí el proveedor que tengas disponible. **Anthropic no tiene modelo de embeddings propio** — si tu proveedor de generación es Anthropic, combinalo con cualquiera de los otros tres para esta parte.

```bash
pip install numpy       # ya instalado si veniás de clases anteriores
pip install openai      # si usás OpenAI
pip install google-genai
pip install ollama
```

In [ ]:
# --- OpenAI ---
from openai import OpenAI

def embed_openai(textos: list[str]) -> list[list[float]]:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.embeddings.create(model=os.environ.get("OPENAI_EMBEDDING_MODEL_ID", "text-embedding-3-large"), input=textos)
    return [d.embedding for d in resp.data]


# --- Google Gemini ---
from google import genai
from google.genai import types

def embed_google(textos: list[str]) -> list[list[float]]:
    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    resp = client.models.embed_content(
        model=os.environ.get("GOOGLE_EMBEDDING_MODEL_ID", "text-embedding-004"),
        contents=textos,
    )
    return [e.values for e in resp.embeddings]


# --- Ollama (local) ---
import ollama

def embed_ollama(textos: list[str]) -> list[list[float]]:
    resp = ollama.embed(model=os.environ.get("OLLAMA_EMBEDDING_MODEL_ID", "nomic-embed-text"), input=textos)
    return resp.embeddings

**🏠 Practicá**: corré la misma lista de `frases` de la sección 2 con la función de tu proveedor y recalculá las similitudes coseno. Los valores exactos van a diferir de los de Bedrock (cada modelo de embeddings vive en su propio espacio vectorial — nunca mezcles embeddings de dos proveedores distintos en el mismo índice), pero el **orden relativo** (frase 1 más parecida a frase 2 que a frase 3) debería mantenerse.

## 4. Limpieza y normalización

Antes de vectorizar, conviene limpiar el texto — no es cosmético, afecta directamente la calidad de la búsqueda:

- **Espacios y saltos de línea redundantes**: texto extraído de PDF/HTML suele traer basura de formato.
- **Boilerplate repetido** (headers/footers, "Página X de Y", menús de navegación): si aparece en todos los chunks, contamina la similitud — todo se parece un poco entre sí por el ruido compartido.
- **Normalización Unicode**: comillas tipográficas, acentos mal codificados, etc.
- **¿Lowercase?** Depende: para texto natural generalmente no hace falta (los embeddings modernos son robustos a mayúsculas), pero para código o identificadores técnicos, bajar a minúsculas puede destruir información relevante (`API_KEY` ≠ `api_key` en un fragmento de código). No lo hagas por default sin pensarlo.

In [ ]:
import re
import unicodedata

def limpiar_texto(texto: str) -> str:
    texto = unicodedata.normalize("NFKC", texto)           # normaliza comillas/acentos
    texto = re.sub(r"[ \t]+", " ", texto)                   # espacios repetidos
    texto = re.sub(r"\n{3,}", "\n\n", texto)                # más de 2 saltos de línea seguidos
    return texto.strip()


texto_sucio = """   Página 1 de 3


Guía  de   configuración


Para configurar  las credenciales,   copiá .env.example a .env.\n\n\n\nCompletá la sección de tu proveedor.
"""

print(repr(limpiar_texto(texto_sucio)))

**🏠 Practicá**: agregá una regla a `limpiar_texto` que detecte y elimine una línea de boilerplate recurrente (ej. `"Página X de Y"` con una regex) — es el tipo de ruido que vas a encontrar en PDFs reales del corpus del TP.

## 5. Estrategias de chunking y solapamiento

Un documento entero casi nunca entra (ni conviene que entre) en un solo embedding — hay que partirlo en **chunks**. Dos decisiones centrales:

- **Tamaño del chunk**: muy chico → pierde contexto (una oración sola puede ser ambigua). Muy grande → mezcla varios temas en un solo vector, diluyendo la similitud con cualquiera de ellos.
- **Solapamiento (overlap)**: sin overlap, una idea que cruza el límite entre dos chunks queda cortada en ambos. Con un poco de overlap (ej. 10-20% del tamaño del chunk), se mitiga ese corte a costa de un poco de redundancia.

In [ ]:
def chunking_por_caracteres(texto: str, tamano: int = 300, solapamiento: int = 50) -> list[str]:
    """Chunking de tamaño fijo con solapamiento, por caracteres (simple, no respeta oraciones)."""
    chunks = []
    paso = tamano - solapamiento
    for inicio in range(0, len(texto), paso):
        chunk = texto[inicio:inicio + tamano]
        if chunk.strip():
            chunks.append(chunk)
        if inicio + tamano >= len(texto):
            break
    return chunks


def chunking_por_parrafos(texto: str, max_chars: int = 500) -> list[str]:
    """Agrupa párrafos (separados por línea en blanco) hasta llegar al tamaño máximo."""
    parrafos = [p.strip() for p in texto.split("\n\n") if p.strip()]
    chunks, actual = [], ""
    for p in parrafos:
        if len(actual) + len(p) + 1 <= max_chars:
            actual = (actual + "\n\n" + p).strip()
        else:
            if actual:
                chunks.append(actual)
            actual = p
    if actual:
        chunks.append(actual)
    return chunks

## 6. Metadata, identificadores y trazabilidad

Un chunk suelto, sin contexto de dónde vino, no sirve para citar fuentes (Clase 6) ni para debuggear por qué el sistema recuperó algo irrelevante (Clase 7). Cada chunk necesita viajar con su metadata.

In [ ]:
import hashlib
from dataclasses import dataclass, field

@dataclass
class Chunk:
    texto: str
    documento_id: str
    posicion: int           # orden del chunk dentro del documento
    chunk_id: str = field(init=False)

    def __post_init__(self):
        # id determinístico: mismo documento + misma posición -> mismo id, siempre
        hash_contenido = hashlib.sha256(self.texto.encode()).hexdigest()[:8]
        self.chunk_id = f"{self.documento_id}-{self.posicion:03d}-{hash_contenido}"


def construir_chunks(texto: str, documento_id: str, tamano: int = 300, solapamiento: int = 50) -> list[Chunk]:
    textos = chunking_por_caracteres(texto, tamano, solapamiento)
    return [Chunk(texto=t, documento_id=documento_id, posicion=i) for i, t in enumerate(textos)]


corpus_ejemplo = """La Converse API de Bedrock es una interfaz unificada para conversación. Funciona igual para
distintos modelos: cambia el modelId, no la forma de llamar. Soporta mensajes de sistema,
parámetros de inferencia como temperatura y top_p, y tool calling con un schema JSON de argumentos.

Para usarla se necesita un cliente de boto3 apuntando al servicio bedrock-runtime, y credenciales
configuradas vía variables de entorno o un perfil de AWS."""

chunks = construir_chunks(limpiar_texto(corpus_ejemplo), documento_id="doc-converse-api")
for c in chunks:
    print(f"[{c.chunk_id}] (pos {c.posicion}) {c.texto[:60]}...")

**🏠 Practicá**: agregá un campo `fuente: str` (ej. una URL o un nombre de archivo) al dataclass `Chunk`, y otro campo `fecha_ingesta` con un timestamp. Esto es exactamente lo que vas a necesitar en la Clase 6 para implementar citas.

## 7. Separación entre ingesta y consulta

Son dos pipelines distintos, que corren en momentos distintos:

- **Ingesta** (lo de hoy): se corre una vez (o cada vez que el corpus cambia) — limpiar, chunkear, embeber, guardar en un índice. Es costoso en tiempo pero infrecuente.
- **Consulta** (Clase 5 en adelante): se corre en cada pregunta del usuario — embeber la consulta y buscar en el índice ya construido. Tiene que ser rápido.

Error común: mezclar la lógica de las dos etapas en una sola función que "ingesta y busca" — hace que no se pueda optimizar cada una por separado (por ejemplo, batchear la ingesta, cachear embeddings de consultas frecuentes).

## 8. Comparación de estrategias de chunking

No hay un tamaño "correcto" — depende del corpus y de qué tipo de pregunta se espera responder. Comparemos en la práctica.

In [ ]:
documento_largo = corpus_ejemplo * 3  # simulamos un documento más largo repitiendo el de ejemplo

for tamano, overlap in [(150, 0), (150, 30), (400, 0), (400, 80)]:
    chunks_resultantes = chunking_por_caracteres(documento_largo, tamano=tamano, solapamiento=overlap)
    print(f"tamano={tamano:>4} overlap={overlap:>3} -> {len(chunks_resultantes)} chunks")

**🏠 Practicá**: corré `chunking_por_parrafos` sobre el mismo `documento_largo` con dos valores de `max_chars` distintos, y compará la cantidad de chunks contra `chunking_por_caracteres`. ¿En qué casos el chunking por párrafos corta "mejor" (en límites de idea) que el de tamaño fijo?

## 9. Chunking semántico (contenido complementario)

Más allá de tamaño fijo o por párrafos, existe el **chunking semántico**: se embeben oraciones consecutivas y se corta el chunk donde la similitud entre una oración y la siguiente cae por debajo de un umbral — la idea es que el corte coincida con un cambio real de tema, no con un límite arbitrario de caracteres.

Es más costoso (requiere embeber a nivel oración durante la ingesta, no solo el chunk final) y no siempre mejora los resultados lo suficiente para justificar el costo — por eso queda como lectura complementaria, no como técnica obligatoria del curso. Si te interesa, buscá "semantic chunking" + el framework que estés usando (ej. LangChain tiene un `SemanticChunker`).

## 10. Laboratorio: pipeline de ingesta documental

**🏠 Tarea de la semana — entregar/completar antes de la Clase 5.**

**Checkpoint del proyecto integrador (Clase 4)**: pipeline de ingesta del corpus (limpieza, chunking y metadata) listo para alimentar el índice vectorial de la Clase 5.

### Consigna

1. Tomá el corpus de referencia provisto por la cátedra (o un subconjunto representativo, para no esperar horas de embeddings si el corpus es grande).
2. Aplicá `limpiar_texto` (o tu propia versión extendida con la practicá de la sección 4) a cada documento.
3. Chunkeá cada documento con la estrategia que prefieras (tamaño fijo o por párrafos) — justificá en un comentario por qué elegiste ese tamaño/overlap para este corpus en particular.
4. Construí un `Chunk` por fragmento, con `documento_id`, `posicion`, y al menos `fuente` (de la practicá de la sección 6).
5. Embebé todos los chunks con la función de tu proveedor y guardá los vectores junto con su `chunk_id` en una estructura simple (una lista de dicts, o un `.json` / `.pkl` — en la Clase 5 esto se migra a un vector store real).

### Criterio de éxito

- Corriendo el pipeline sobre el corpus completo (o el subconjunto elegido) no tira ningún error.
- Cada chunk resultante tiene menos de ~1500 caracteres (si no, revisá el tamaño elegido).
- Podés tomar cualquier `chunk_id` y recuperar de qué documento y qué posición vino, sin ambigüedad.
- El código de ingesta no incluye ninguna lógica de "búsqueda" — eso es, a propósito, la Clase 5.

In [ ]:
def pipeline_ingesta(documentos: dict[str, str], tamano=300, solapamiento=50, embed_fn=embed_bedrock) -> list[dict]:
    """documentos: {documento_id: texto_crudo}. Devuelve una lista de {chunk_id, texto, vector, documento_id, posicion}."""
    # TODO: para cada documento, limpiar -> chunkear -> construir Chunk -> embeber (en batch, no uno por uno)
    # TODO: devolver la lista combinando los datos del Chunk con su vector correspondiente
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# resultado = pipeline_ingesta({"doc-converse-api": corpus_ejemplo})
# print(len(resultado), "chunks ingeridos")

## Cierre

- Hoy construimos la mitad "offline" de un sistema RAG: limpieza, chunking con criterio, metadata trazable y embeddings — sin tocar todavía la parte de búsqueda.
- Próxima clase: vector stores — ahí vive el índice que el pipeline de hoy alimenta, y ahí empieza la mitad "online" (la consulta).
- Antes de la próxima clase: dejar el pipeline de la sección 10 funcionando sobre el corpus del TP.